In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoConfig
from pathlib import Path

device = 'cpu'
dtype = torch.float32

model_id="Qwen/Qwen3-0.6B"


# load base model
model_config = AutoConfig.from_pretrained(
    model_id,
)


base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    config=model_config,
    device_map=device,
    torch_dtype=dtype,
)

`torch_dtype` is deprecated! Use `dtype` instead!


In [7]:
# get a linear layer weight for SVD
for name, param in base_model.named_modules():
    if isinstance(param, torch.nn.Linear):
        print(name)
        base_weight = param
        break
print(name)
W = base_weight.weight

U, S, Vh = torch.linalg.svd(W.float(), full_matrices=False)

model.layers.0.self_attn.q_proj
model.layers.0.self_attn.q_proj


In [10]:
def reconstruction_error(W_orig, W_recon):
    """Compute relative Frobenius norm error"""
    return torch.norm(W_orig - W_recon) / torch.norm(W_orig)

# Original reconstruction (full precision float32)
W_recon_fp32 = U @ torch.diag(S) @ Vh
err_fp32 = reconstruction_error(W, W_recon_fp32)
print(f"FP32 reconstruction error: {err_fp32:.2e}")

# Test different dtypes
dtypes_to_test = [
    ('bfloat16', torch.bfloat16),
    ('float16', torch.float16),
]

print("\n--- Dtype quantization ---")
for name, dtype in dtypes_to_test:
    U_q = U.to(dtype).to(torch.float32)
    S_q = S.to(dtype).to(torch.float32)
    Vh_q = Vh.to(dtype).to(torch.float32)
    
    W_recon = U_q @ torch.diag(S_q) @ Vh_q
    err = reconstruction_error(W, W_recon)
    print(f"{name:10s}: {err:.2e} (vs fp32: {err/err_fp32:.2f}x)")

# Test int8 quantization (symmetric, per-tensor)
print("\n--- Int8 quantization (symmetric) ---")
def quantize_int8(tensor):
    """Symmetric int8 quantization"""
    scale = tensor.abs().max() / 127
    q = torch.clamp(torch.round(tensor / scale), -128, 127).to(torch.int8)
    return q, scale

def dequantize_int8(q_tensor, scale):
    return q_tensor.to(torch.float32) * scale

def quantize_int4(tensor):
    """Symmetric int4 quantization"""
    scale = tensor.abs().max() / 7
    q = torch.clamp(torch.round(tensor / scale), -8, 7).to(torch.int8)  # store in int8
    return q, scale

def dequantize_int4(q_tensor, scale):
    return q_tensor.to(torch.float32) * scale

for component_name, component in [('U', U), ('S', S), ('Vh', Vh)]:
    U_use, S_use, Vh_use = U.clone(), S.clone(), Vh.clone()
    
    q, scale = quantize_int8(component)
    dq = dequantize_int8(q, scale)
    
    if component_name == 'U':
        U_use = dq
    elif component_name == 'S':
        S_use = dq
    else:
        Vh_use = dq
    
    W_recon = U_use @ torch.diag(S_use) @ Vh_use
    err = reconstruction_error(W, W_recon)
    print(f"int8 {component_name:2s}:     {err:.2e} (vs fp32: {err/err_fp32:.2f}x)")

# Test all components int8
print("\n--- All components int8 ---")
U_q, U_scale = quantize_int8(U)
S_q, S_scale = quantize_int8(S)
Vh_q, Vh_scale = quantize_int8(Vh)

U_dq = dequantize_int8(U_q, U_scale)
S_dq = dequantize_int8(S_q, S_scale)
Vh_dq = dequantize_int8(Vh_q, Vh_scale)

W_recon = U_dq @ torch.diag(S_dq) @ Vh_dq
err = reconstruction_error(W, W_recon)
print(f"All int8:    {err:.2e} (vs fp32: {err/err_fp32:.2f}x)")

# Test int4 (just for S since it's 1D and most important)
print("\n--- Int4 quantization (S only) ---")
S_q4, S_scale4 = quantize_int4(S)
S_dq4 = S_q4.to(torch.float32) * S_scale4

W_recon = U @ torch.diag(S_dq4) @ Vh
err = reconstruction_error(W, W_recon)
print(f"int4 S:      {err:.2e} (vs fp32: {err/err_fp32:.2f}x)")

FP32 reconstruction error: 2.03e-06

--- Dtype quantization ---
bfloat16  : 2.90e-03 (vs fp32: 1430.18x)
float16   : 3.69e-04 (vs fp32: 181.59x)

--- Int8 quantization (symmetric) ---
int8 U :     5.78e-02 (vs fp32: 28477.35x)
int8 S :     1.57e-02 (vs fp32: 7753.11x)
int8 Vh:     5.73e-02 (vs fp32: 28226.64x)

--- All components int8 ---
All int8:    8.30e-02 (vs fp32: 40904.29x)

--- Int4 quantization (S only) ---
int4 S:      3.12e-01 (vs fp32: 153782.02x)


In [11]:
# Compare: Direct quantization vs SVD quantization
print("\n" + "="*60)
print("COMPARISON: Direct W quantization vs SVD component quantization")
print("="*60)

# Direct int8 quantization of full weight matrix
print("\n--- Direct weight quantization ---")
W_q8, W_scale8 = quantize_int8(W)
W_dq8 = dequantize_int8(W_q8, W_scale8)
err_direct_int8 = reconstruction_error(W, W_dq8)
print(f"Direct int8:     {err_direct_int8:.2e}")

# Direct int4 quantization
W_q4, W_scale4 = quantize_int4(W)
W_dq4 = dequantize_int4(W_q4, W_scale4)
err_direct_int4 = reconstruction_error(W, W_dq4)
print(f"Direct int4:     {err_direct_int4:.2e}")

# SVD int8 (all components)
err_svd_int8 = err  # from previous cell
print(f"\nSVD all int8:    {err_svd_int8:.2e}")
print(f"Ratio (SVD/Direct): {err_svd_int8/err_direct_int8:.2f}x worse")

# SVD int4 on all components
print("\n--- SVD all int4 ---")
U_q4, U_scale4 = quantize_int4(U)
S_q4, S_scale4 = quantize_int4(S)
Vh_q4, Vh_scale4 = quantize_int4(Vh)

U_dq4 = dequantize_int4(U_q4, U_scale4)
S_dq4 = dequantize_int4(S_q4, S_scale4)
Vh_dq4 = dequantize_int4(Vh_q4, Vh_scale4)

W_recon_svd_int4 = U_dq4 @ torch.diag(S_dq4) @ Vh_dq4
err_svd_int4 = reconstruction_error(W, W_recon_svd_int4)
print(f"SVD all int4:    {err_svd_int4:.2e}")
print(f"Ratio (SVD/Direct): {err_svd_int4/err_direct_int4:.2f}x worse")

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Direct int8:     {err_direct_int8:.2e}")
print(f"SVD int8:        {err_svd_int8:.2e}  ({err_svd_int8/err_direct_int8:.1f}x worse)")
print(f"Direct int4:     {err_direct_int4:.2e}")
print(f"SVD int4:        {err_svd_int4:.2e}  ({err_svd_int4/err_direct_int4:.1f}x worse)")
print("\nConclusion: SVD quantization is {'MORE' if err_svd_int4/err_direct_int4 > 1 else 'LESS'} sensitive than direct quantization")


COMPARISON: Direct W quantization vs SVD component quantization

--- Direct weight quantization ---
Direct int8:     4.67e-02
Direct int4:     7.48e-01

SVD all int8:    3.12e-01
Ratio (SVD/Direct): 6.68x worse

--- SVD all int4 ---
SVD all int4:    1.10e+00
Ratio (SVD/Direct): 1.47x worse

SUMMARY
Direct int8:     4.67e-02
SVD int8:        3.12e-01  (6.7x worse)
Direct int4:     7.48e-01
SVD int4:        1.10e+00  (1.5x worse)

Conclusion: SVD quantization is {'MORE' if err_svd_int4/err_direct_int4 > 1 else 'LESS'} sensitive than direct quantization


## Practical SVD quantization strategies

For actual compression, you'd likely:
1. Keep S (singular values) in high precision - they're small (1D) and critical
2. Quantize U and Vh (orthogonal matrices) - they're large but more robust to quantization

Let's test different strategies:

In [12]:
print("="*60)
print("PRACTICAL SVD QUANTIZATION STRATEGIES")
print("="*60)

# Strategy 1: Quantize U and Vh to int8, keep S in fp32
print("\n--- Strategy 1: U,Vh int8 + S fp32 ---")
U_q8, U_s8 = quantize_int8(U)
Vh_q8, Vh_s8 = quantize_int8(Vh)
U_dq8 = dequantize_int8(U_q8, U_s8)
Vh_dq8 = dequantize_int8(Vh_q8, Vh_s8)

W_recon = U_dq8 @ torch.diag(S) @ Vh_dq8  # S stays fp32
err_strat1 = reconstruction_error(W, W_recon)
print(f"Error: {err_strat1:.2e} (vs direct int8: {err_strat1/err_direct_int8:.2f}x)")

# Strategy 2: Quantize U and Vh to int4, keep S in fp32
print("\n--- Strategy 2: U,Vh int4 + S fp32 ---")
U_q4, U_s4 = quantize_int4(U)
Vh_q4, Vh_s4 = quantize_int4(Vh)
U_dq4 = dequantize_int4(U_q4, U_s4)
Vh_dq4 = dequantize_int4(Vh_q4, Vh_s4)

W_recon = U_dq4 @ torch.diag(S) @ Vh_dq4  # S stays fp32
err_strat2 = reconstruction_error(W, W_recon)
print(f"Error: {err_strat2:.2e} (vs direct int4: {err_strat2/err_direct_int4:.2f}x)")

# Strategy 3: Quantize U and Vh to int4, S to fp16 (even more aggressive)
print("\n--- Strategy 3: U,Vh int4 + S fp16 ---")
S_fp16 = S.to(torch.float16).to(torch.float32)
W_recon = U_dq4 @ torch.diag(S_fp16) @ Vh_dq4
err_strat3 = reconstruction_error(W, W_recon)
print(f"Error: {err_strat3:.2e} (vs direct int4: {err_strat3/err_direct_int4:.2f}x)")

# Strategy 4: U,Vh bfloat16 + S fp32 (less aggressive, more practical)
print("\n--- Strategy 4: U,Vh bf16 + S fp32 ---")
U_bf16 = U.to(torch.bfloat16).to(torch.float32)
Vh_bf16 = Vh.to(torch.bfloat16).to(torch.float32)
W_recon = U_bf16 @ torch.diag(S) @ Vh_bf16
err_strat4 = reconstruction_error(W, W_recon)
print(f"Error: {err_strat4:.2e} (vs direct bf16: {err_strat4/reconstruction_error(W, W.to(torch.bfloat16).to(torch.float32)):.2f}x)")

print("\n" + "="*60)
print("MEMORY FOOTPRINT COMPARISON (for this layer)")
print("="*60)
# Calculate memory for each approach
h, o = W.shape
fp32_size = h * o * 4  # 4 bytes per param
int8_size = h * o * 1 + 4  # 1 byte per param + scale
int4_size = h * o * 0.5 + 4  # 0.5 bytes per param + scale

# SVD components
svd_fp32 = (h * min(h,o) + min(h,o) + min(h,o) * o) * 4
svd_strat1 = (h * min(h,o) * 1 + 4) + min(h,o) * 4 + (min(h,o) * o * 1 + 4)  # U int8, S fp32, Vh int8
svd_strat2 = (h * min(h,o) * 0.5 + 4) + min(h,o) * 4 + (min(h,o) * o * 0.5 + 4)  # U int4, S fp32, Vh int4

print(f"Original fp32:           {fp32_size/1024:.1f} KB")
print(f"Direct int8:             {int8_size/1024:.1f} KB ({fp32_size/int8_size:.2f}x smaller)")
print(f"Direct int4:             {int4_size/1024:.1f} KB ({fp32_size/int4_size:.2f}x smaller)")
print(f"\nSVD fp32:                {svd_fp32/1024:.1f} KB ({fp32_size/svd_fp32:.2f}x vs original)")
print(f"SVD Strategy 1 (U,Vh int8 + S fp32): {svd_strat1/1024:.1f} KB ({fp32_size/svd_strat1:.2f}x smaller)")
print(f"SVD Strategy 2 (U,Vh int4 + S fp32): {svd_strat2/1024:.1f} KB ({fp32_size/svd_strat2:.2f}x smaller)")

print("\n" + "="*60)
print("SUMMARY: Error vs Compression tradeoff")
print("="*60)
print(f"Direct int8:     {err_direct_int8:.2e}  ({fp32_size/int8_size:.2f}x compression)")
print(f"SVD U,Vh int8:   {err_strat1:.2e}  ({fp32_size/svd_strat1:.2f}x compression)")
print(f"Direct int4:     {err_direct_int4:.2e}  ({fp32_size/int4_size:.2f}x compression)")
print(f"SVD U,Vh int4:   {err_strat2:.2e}  ({fp32_size/svd_strat2:.2f}x compression)")
print(f"\nConclusion: SVD with selective quantization (keep S high precision)")
print(f"            {'BETTER' if err_strat2 < err_direct_int4 else 'WORSE'} than direct quantization at similar compression")

PRACTICAL SVD QUANTIZATION STRATEGIES

--- Strategy 1: U,Vh int8 + S fp32 ---
Error: 8.15e-02 (vs direct int8: 1.74x)

--- Strategy 2: U,Vh int4 + S fp32 ---
Error: 1.06e+00 (vs direct int4: 1.42x)

--- Strategy 3: U,Vh int4 + S fp16 ---
Error: 1.06e+00 (vs direct int4: 1.42x)

--- Strategy 4: U,Vh bf16 + S fp32 ---
Error: 2.34e-03 (vs direct bf16: infx)

MEMORY FOOTPRINT COMPARISON (for this layer)
Original fp32:           8192.0 KB
Direct int8:             2048.0 KB (4.00x smaller)
Direct int4:             1024.0 KB (8.00x smaller)

SVD fp32:                12292.0 KB (0.67x vs original)
SVD Strategy 1 (U,Vh int8 + S fp32): 3076.0 KB (2.66x smaller)
SVD Strategy 2 (U,Vh int4 + S fp32): 1540.0 KB (5.32x smaller)

SUMMARY: Error vs Compression tradeoff
Direct int8:     4.67e-02  (4.00x compression)
SVD U,Vh int8:   8.15e-02  (2.66x compression)
Direct int4:     7.48e-01  (8.00x compression)
SVD U,Vh int4:   1.06e+00  (5.32x compression)

Conclusion: SVD with selective quantization (kee

## Sparsifying U and V

Can we zero out small entries in U and V to reduce memory?

In [13]:
print("="*60)
print("SPARSIFYING U AND V (magnitude pruning)")
print("="*60)

def sparsify_topk(tensor, keep_fraction):
    """Keep only top-k% of entries by magnitude, zero out rest."""
    flat = tensor.abs().flatten()
    k = int(flat.numel() * keep_fraction)
    threshold = torch.topk(flat, k).values.min()
    mask = tensor.abs() >= threshold
    return tensor * mask, mask.float().mean().item()

# Test different sparsity levels
sparsity_levels = [1.0, 0.5, 0.25, 0.1, 0.05]

print("\nSparsity | Error      | U nnz | V nnz | Total params")
print("-" * 60)

for keep_frac in sparsity_levels:
    U_sparse, u_nnz = sparsify_topk(U, keep_frac)
    V_sparse, v_nnz = sparsify_topk(Vh, keep_frac)
    
    # Reconstruct
    W_recon = U_sparse @ torch.diag(S) @ V_sparse
    err = reconstruction_error(W, W_recon)
    
    # Calculate effective parameters (assuming sparse storage)
    h, o = W.shape
    r = S.shape[0]
    u_params = int(h * r * u_nnz)
    v_params = int(r * o * v_nnz)
    s_params = r
    total_params = u_params + v_params + s_params
    orig_params = h * o
    
    print(f"{keep_frac:6.2f}   | {err:.2e} | {u_nnz:.2%} | {v_nnz:.2%} | {total_params:6d} ({total_params/orig_params:.2%} of orig)")

print("\n" + "="*60)
print("COMBINED: Sparsify U,V + Quantize to int8")
print("="*60)

for keep_frac in [0.5, 0.25, 0.1]:
    U_sparse, u_nnz = sparsify_topk(U, keep_frac)
    V_sparse, v_nnz = sparsify_topk(Vh, keep_frac)
    
    # Quantize the sparse matrices
    U_q, U_scale = quantize_int8(U_sparse)
    V_q, V_scale = quantize_int8(V_sparse)
    U_dq = dequantize_int8(U_q, U_scale)
    V_dq = dequantize_int8(V_q, V_scale)
    
    # Reconstruct
    W_recon = U_dq @ torch.diag(S) @ V_dq
    err = reconstruction_error(W, W_recon)
    
    # Memory: sparse int8 storage
    h, o = W.shape
    r = S.shape[0]
    u_bytes = int(h * r * u_nnz * 1)  # 1 byte per int8
    v_bytes = int(r * o * v_nnz * 1)
    s_bytes = r * 4  # S stays fp32
    total_bytes = u_bytes + v_bytes + s_bytes
    orig_bytes = h * o * 4
    
    print(f"Sparsity {keep_frac:.2f}: {err:.2e} error, {total_bytes/1024:.1f}KB ({total_bytes/orig_bytes:.2%} of orig fp32)")

SPARSIFYING U AND V (magnitude pruning)

Sparsity | Error      | U nnz | V nnz | Total params
------------------------------------------------------------
  1.00   | 2.03e-06 | 100.00% | 100.00% | 3146752 (150.05% of orig)
  0.50   | 3.54e-01 | 50.00% | 50.00% | 1573888 (75.05% of orig)
  0.25   | 6.58e-01 | 25.00% | 25.00% | 787456 (37.55% of orig)
  0.10   | 8.67e-01 | 10.00% | 10.00% | 315596 (15.05% of orig)
  0.05   | 9.36e-01 | 5.00% | 5.00% | 158309 (7.55% of orig)

COMBINED: Sparsify U,V + Quantize to int8
Sparsity 0.50: 3.58e-01 error, 1540.0KB (18.80% of orig fp32)
Sparsity 0.25: 6.59e-01 error, 772.0KB (9.42% of orig fp32)
Sparsity 0.10: 8.67e-01 error, 311.2KB (3.80% of orig fp32)


## Rank reduction: Keep top-k singular values

Instead of sparsifying by magnitude, use SVD's natural compression: keep only top-k singular values.

In [14]:
print("="*60)
print("RANK REDUCTION: Keep top-k singular values")
print("="*60)

# Test different ranks
full_rank = S.shape[0]
ranks_to_test = [full_rank, full_rank//2, full_rank//4, full_rank//8, 100, 50, 20]

print("\nRank | Error      | Params    | Compression | Explained variance")
print("-" * 75)

total_variance = (S ** 2).sum()

for r in ranks_to_test:
    if r > full_rank:
        continue
    
    # Truncate to rank r
    U_r = U[:, :r]
    S_r = S[:r]
    Vh_r = Vh[:r, :]
    
    # Reconstruct
    W_recon = U_r @ torch.diag(S_r) @ Vh_r
    err = reconstruction_error(W, W_recon)
    
    # Calculate parameters
    h, o = W.shape
    params = h * r + r + r * o  # U + S + Vh
    orig_params = h * o
    compression = orig_params / params
    
    # Explained variance
    explained_var = (S_r ** 2).sum() / total_variance
    
    print(f"{r:4d} | {err:.2e} | {params:8d} | {compression:6.2f}x    | {explained_var:.2%}")

print("\n" + "="*60)
print("RANK REDUCTION + QUANTIZATION")
print("="*60)

# Test rank reduction with quantization
for r in [full_rank//2, full_rank//4, 100]:
    if r > full_rank:
        continue
    
    U_r = U[:, :r]
    S_r = S[:r]
    Vh_r = Vh[:r, :]
    
    # Quantize U and Vh to int8, keep S in fp32
    U_q, U_scale = quantize_int8(U_r)
    Vh_q, Vh_scale = quantize_int8(Vh_r)
    U_dq = dequantize_int8(U_q, U_scale)
    Vh_dq = dequantize_int8(Vh_q, Vh_scale)
    
    W_recon = U_dq @ torch.diag(S_r) @ Vh_dq
    err = reconstruction_error(W, W_recon)
    
    # Memory calculation
    h, o = W.shape
    u_bytes = h * r * 1  # int8
    v_bytes = r * o * 1  # int8
    s_bytes = r * 4      # fp32
    total_bytes = u_bytes + v_bytes + s_bytes + 8  # +8 for scales
    orig_bytes = h * o * 4
    
    explained_var = (S_r ** 2).sum() / total_variance
    
    print(f"Rank {r:4d}: {err:.2e} error, {total_bytes/1024:.1f}KB ({total_bytes/orig_bytes:.2%} of orig), {explained_var:.1%} variance")

print("\n" + "="*60)
print("COMPARISON: Rank reduction vs Direct quantization")
print("="*60)
print(f"Direct int8:        {err_direct_int8:.2e} error, {int8_size/1024:.1f}KB")
print(f"Rank {full_rank//4} + int8:   {err:.2e} error, {total_bytes/1024:.1f}KB")
print(f"\nRank reduction gives better compression but {'WORSE' if err > err_direct_int8 else 'BETTER'} quality")

RANK REDUCTION: Keep top-k singular values

Rank | Error      | Params    | Compression | Explained variance
---------------------------------------------------------------------------
1024 | 2.03e-06 |  3146752 |   0.67x    | 100.00%
 512 | 3.92e-01 |  1573376 |   1.33x    | 84.62%
 256 | 6.27e-01 |   786688 |   2.67x    | 60.68%
 128 | 7.69e-01 |   393344 |   5.33x    | 40.89%
 100 | 8.03e-01 |   307300 |   6.82x    | 35.49%
  50 | 8.69e-01 |   153650 |  13.65x    | 24.43%
  20 | 9.15e-01 |    61460 |  34.12x    | 16.32%

RANK REDUCTION + QUANTIZATION
Rank  512: 3.96e-01 error, 1538.0KB (18.77% of orig), 84.6% variance
Rank  256: 6.29e-01 error, 769.0KB (9.39% of orig), 60.7% variance
Rank  100: 8.04e-01 error, 300.4KB (3.67% of orig), 35.5% variance

COMPARISON: Rank reduction vs Direct quantization
Direct int8:        4.67e-02 error, 2048.0KB
Rank 256 + int8:   8.04e-01 error, 300.4KB

Rank reduction gives better compression but WORSE quality


## QLoRA-style test: Quantized base + full precision adapter

QLoRA doesn't quantize the adapter - it quantizes the base weight W to 4-bit, then trains a full-precision LoRA adapter on top. Let's test if SVD decomposition helps here.